# 05. 전이학습

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/2038prob/defense-ai-textbook-code/blob/main/notebooks/%EC%8B%A4%EC%8A%B5_05_%EC%A0%84%EC%9D%B4%ED%95%99%EC%8A%B5.ipynb)

「국방 인공지능」 실습 코드 — 이 장의 파이썬 코드를 순서대로 모았습니다. **맨 위 setup 셀부터 위에서 아래로** 실행하세요(앞 셀에서 정의한 변수를 뒤 셀이 이어 씁니다). ⚠️ 표시가 붙은 셀은 개념 설명용 조각으로, 그대로는 실행되지 않을 수 있습니다.

전체 저장소: https://github.com/2038prob/defense-ai-textbook-code


In [ ]:
# === 실습 환경 준비 (Colab/로컬 공통) — 가장 먼저 실행하세요 ===
# 필요한 패키지를 설치하고(이미 있으면 건너뜀) 공통 유틸을 정의합니다.
import importlib.util, subprocess, sys
for pkg in ["torch", "torchvision", "numpy", "matplotlib", "scipy"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

import random, numpy as np, torch

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", device)

## 실전 주의점과 수식

### 🔬 실습 — L2-SP를 직접 구현하고, 전이 효과를 대조 실험으로 확인하기

① **L2-SP 구현**: 사전학습 가중치를 복사해 두고, 매 스텝 거리 패널티를 더한다.


In [ ]:
import copy
import torch
import torch.nn as nn
from torchvision import models

model = models.resnet18(weights='IMAGENET1K_V1')
model.fc = nn.Linear(model.fc.in_features, 4)   # 4클래스 헤드 교체

# 출발점(사전학습 가중치) 스냅숏 -- 백본만, 기울기 추적 없이 보관
theta0 = {n: p.detach().clone()
          for n, p in model.named_parameters() if not n.startswith('fc')}

def l2sp_penalty(model, theta0, lam=1e-3):
    pen = 0.0
    for n, p in model.named_parameters():
        if n in theta0:                          # 백본: 출발점으로 당김
            pen = pen + ((p - theta0[n]) ** 2).sum()
    return lam * pen

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# 학습 스텝: 기존 손실에 한 항만 추가된다
def train_step(x, y):
    loss = criterion(model(x), y) + l2sp_penalty(model, theta0)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    return loss.item()

## 층별 학습률과 단계적 동결 해제

### 📝 예제 — 층별 학습률(Discriminative LR) 계산

④ 즉 출력에 가까운 분류기는 빠르게, 입력에 가까운 범용 특징층은 거의 고정에 가깝게 갱신한다. PyTorch에서는 `optimizer`의 `param_groups`로 층별 LR을 지정한다.


In [ ]:
optimizer = torch.optim.Adam([
    {"params": model.layer1.parameters(), "lr": 0.012e-3},
    {"params": model.layer2.parameters(), "lr": 0.037e-3},
    {"params": model.layer3.parameters(), "lr": 0.11e-3},
    {"params": model.layer4.parameters(), "lr": 0.33e-3},
    {"params": model.fc.parameters(),     "lr": 1.0e-3},
])

## 실습: 소량 국방 차량 영상 전이학습

### 🔬 실습 — ResNet-18 전이학습 (Feature Extractor $$ 단계적 Fine-tuning)

① **데이터 준비** — 사전학습 입력 규격(224, ImageNet 정규화)에 맞추고 소량 데이터를 증강으로 보완.


> ℹ️ **실습용 합성 데이터** — 실제 국방 차량 이미지가 없어도 노트북이 실행되도록, 아래 셀이 `dataset/military_vehicles/train/<클래스>/` 구조와 임의 이미지를 생성합니다. (교재의 `ImageFolder` 사용법을 그대로 확인할 수 있습니다.)

In [ ]:
# --- (실습용 합성 데이터) 실제 데이터가 없을 때 폴더 구조와 임의 이미지를 생성 ---
import os
import numpy as np
from PIL import Image

_root = "dataset/military_vehicles/train"
for _cls in ["tank", "apc", "truck", "artillery", "other"]:
    _d = os.path.join(_root, _cls)
    os.makedirs(_d, exist_ok=True)
    for _i in range(8):                       # 클래스당 8장
        _arr = (np.random.rand(224, 224, 3) * 255).astype("uint8")
        Image.fromarray(_arr).save(os.path.join(_d, f"img{_i:03d}.jpg"))
print("합성 데이터 준비 완료:", _root)

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

train_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),  # ImageNet
])
train_set = datasets.ImageFolder("dataset/military_vehicles/train", transform=train_tf)
train_loader = DataLoader(train_set, batch_size=16, shuffle=True)   # 소량 -> 작은 배치

코드의 `datasets.ImageFolder`는 별도 레이블 파일 없이 *폴더 이름*을 클래스로 사용한다. 즉 데이터를 다음처럼 "클래스별 하위 폴더"로 정리해 두면 된다.

> ⚠️ **설명용 조각** — 아래는 개념을 보여 주는 코드로, 그대로는 실행되지 않을 수 있습니다.


In [ ]:
dataset/military_vehicles/train/
    tank/        img001.jpg, img002.jpg, ...
    apc/         ...
    truck/       ...
    artillery/   ...
    other/       ...

② **모델 준비** — 사전학습 ResNet-18의 마지막 FC를 5클래스용으로 교체.


In [ ]:
import torch, torch.nn as nn
from torchvision import models

NUM_CLASSES = 5
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

③ **1단계 — Backbone 동결, Head만 학습** — 분류기가 랜덤 초기값에서 안정화되도록.


In [ ]:
def freeze_backbone(m):
    for name, p in m.named_parameters():
        p.requires_grad = ("fc" in name)        # fc만 학습

freeze_backbone(model)
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"학습 파라미터 수: {n_train:,}")          # 약 2,565 (= 512*5 + 5)
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

④ **2단계 — 단계적 동결 해제** — layer4 $$ layer3 순으로 풀며 LR을 낮춘다.

> ⚠️ **설명용 조각** — 아래는 개념을 보여 주는 코드로, 그대로는 실행되지 않을 수 있습니다.


In [ ]:
def unfreeze(m, layer):                         # 특정 블록 해제
    for name, p in m.named_parameters():
        if layer in name: p.requires_grad = True

unfreeze(model, "layer4")
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
# ... (train_loop 후) layer3 추가 해제, lr=5e-5 로 재학습

---

실습 코드 저장소: https://github.com/2038prob/defense-ai-textbook-code  ·  교재 「국방 인공지능」
